# FoxTrack 04: Time confound test

In notebook 03, the gradient-boosted trees gave alerts on benign packets long before the attack start in two captures. The probable cause: in each capture the attack comes late, so a model can learn the time in the capture and not the attack.

This notebook tests that explanation. Set `VARIANT` and run all cells. Each variant is one run.

| `VARIANT` | What it does |
|---|---|
| `clock` | Trains a model on benign packets only, to tell early packets from late packets. Tests it on a capture that it did not see. A high ROC-AUC shows that the features contain a clock. Also compares the important features of the clock model and the attack model. |
| `drop_clock` | Repeats notebook 03 for the trees, without the features that the clock model uses most. |
| `time_matched` | Repeats notebook 03 for the trees, with benign training packets only from the attack period. Benign and malicious packets then come from the same time, so the time gives no information. |

**Before you run it:** add the output of notebook 01 in **Input > Add Input > Notebook Output**, and set **Internet** to on.

In [1]:
# Settings.
VARIANT = "clock"                      # "clock", "drop_clock", or "time_matched"
REPO = "DiscM/DATA-226-FoxTrack"
BRANCH = "main"
OUT_DIR = "/kaggle/working/confound"
N_PER_CLASS = 100_000
GBT = {"maxIter": 50, "maxDepth": 5}
CLOCK_SHARE = 0.8                      # drop the features that have this share of the clock importance
TARGET_FPRS = (1e-2, 1e-3, 1e-4)
ALERT_RULES = ((1, 1), (3, 5), (10, 20))
assert VARIANT in ("clock", "drop_clock", "time_matched")

In [2]:
import shutil, subprocess, sys
if shutil.which("java") is None:
    subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless", shell=True, check=True)
!pip install -q pyspark mlflow

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None  # a public repository does not need a token
auth = f"{token}@" if token else ""
# A new clone on each run, so the code is always the latest version.
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
done = subprocess.run(
    ["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
     f"https://{auth}github.com/{REPO}.git", "/kaggle/working/repo"],
    capture_output=True, text=True,
)
if done.returncode:
    raise RuntimeError("git clone failed. Examine REPO, BRANCH, and the GITHUB_TOKEN secret.")
sys.path.insert(0, "/kaggle/working/repo/src")
from foxtrack import etl, split, supervised
print("Code version:", subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())

Code version: e6f9ea8 Add the time confound test for the supervised models


In [3]:
# Find the Parquet output of notebook 01.
from pathlib import Path
found = sorted(Path("/kaggle/input").rglob("summary.json")) + sorted(Path("/kaggle/working").glob("*/summary.json"))
if not found:
    raise FileNotFoundError("summary.json not found. Add the output of notebook 01 in Input > Add Input > Notebook Output.")
PARQUET_ROOT = found[0].parent
print("Parquet data:", PARQUET_ROOT)

Parquet data: /kaggle/input/notebooks/<kaggle-user>/notebookabc04dfe5a/kitsune_parquet


In [4]:
import os, time
import pandas as pd
from foxtrack.anomaly import feature_columns

spark = etl.build_spark(driver_memory="20g")
summary = split.load_summary(PARQUET_ROOT)
os.makedirs(f"{OUT_DIR}/scores", exist_ok=True)
ALL_FEATURES = feature_columns(spark.read.parquet(f"{PARQUET_ROOT}/features/attack={summary[0]['attack']}"))
features = ALL_FEATURES
print("Variant:", VARIANT, "| features:", len(ALL_FEATURES))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/22 00:55:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Variant: clock | features: 115


## 1. Clock model

Benign packets before the attack only. Label 1 = the last 15% of the benign period.

In [5]:
%%time
if VARIANT in ("clock", "drop_clock"):
    path = f"{OUT_DIR}/clock_sample"
    supervised.build_clock_sample(spark, PARQUET_ROOT, summary, N_PER_CLASS).write.mode("overwrite").parquet(path)
    clock = supervised.assemble(spark.read.parquet(path)).cache()
    clock_model = supervised.make_estimator("gbt", **GBT).fit(clock)
    clock_importance = supervised.feature_importance(clock_model, ALL_FEATURES)
    clock_importance.to_csv(f"{OUT_DIR}/clock_importance.csv")
    print("### CLOCK_IMPORTANCE"); print(clock_importance.head(25).round(4).to_csv())

if VARIANT == "clock":
    rows = []
    for row in summary:
        start = time.time()
        auc = supervised.clock_fold_auc(clock, row["attack"], maxIter=20, maxDepth=5)
        rows.append({"attack": row["attack"], "clock_roc_auc": auc})
        print(f"{row['attack']}: clock ROC-AUC {auc:.3f} ({time.time() - start:.0f} s)")
    clock_auc = pd.DataFrame(rows)
    clock_auc.to_csv(f"{OUT_DIR}/clock_auc.csv", index=False)
    print("### CLOCK_AUC"); print(clock_auc.to_csv(index=False))

if VARIANT == "drop_clock":
    dropped = supervised.top_features(clock_importance, share=CLOCK_SHARE)
    features = [f for f in ALL_FEATURES if f not in dropped]
    print("### DROPPED", ",".join(dropped))
    print(f"Dropped {len(dropped)} clock features. {len(features)} features remain.")

26/11/22 00:55:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


### CLOCK_IMPORTANCE
,importance
f76,0.1951
f79,0.1947
f78,0.1724
f61,0.0575
f12,0.0414
f62,0.0369
f112,0.0338
f64,0.0314
f63,0.0298
f58,0.027
f54,0.0241
f110,0.0217
f75,0.0207
f14,0.0185
f113,0.0162
f13,0.0121
f101,0.0106
f108,0.0063
f60,0.0057
f105,0.0044
f111,0.0043
f59,0.0043
f55,0.0036
f73,0.0031
f109,0.003

arp_mitm: clock ROC-AUC 0.996 (102 s)
active_wiretap: clock ROC-AUC 0.982 (97 s)
fuzzing: clock ROC-AUC 0.906 (96 s)
mirai_botnet: clock ROC-AUC 0.599 (109 s)
os_scan: clock ROC-AUC 0.902 (97 s)
ssdp_flood: clock ROC-AUC 0.857 (95 s)
ssl_renegotiation: clock ROC-AUC 0.123 (93 s)
syn_dos: clock ROC-AUC 0.991 (101 s)
video_injection: clock ROC-AUC 0.783 (94 s)
### CLOCK_AUC
attack,clock_roc_auc
arp_mitm,0.9957688197000001
active_wiretap,0.9821081588
fuzzing,0.90605048985
mirai_botnet,0.5988397316926113
os_scan,0.90231765375
ssdp_flood,0.8571276597
ssl_renegotiation,0.12322835065000001
syn_dos,0.9906727923499999
video_injection,0.7826348069

CPU times: user 9.49 s, sys: 2.02 s, t

## 2. Attack model

For `clock`: one attack model on all captures, to compare its important features with the clock model.
For the other variants: the leave-one-capture-out test of notebook 03.

In [6]:
%%time
period = "attack" if VARIANT == "time_matched" else "all"
sample_path = f"{OUT_DIR}/train_sample"
supervised.build_training_sample(spark, PARQUET_ROOT, summary, N_PER_CLASS, benign_period=period).write.mode("overwrite").parquet(sample_path)
train = supervised.assemble(spark.read.parquet(sample_path), features).cache()
train.groupBy("attack").pivot("label").count().orderBy("attack").show()

if VARIANT == "clock":
    attack_model = supervised.make_estimator("gbt", **GBT).fit(train)
    attack_importance = supervised.feature_importance(attack_model, ALL_FEATURES)
    attack_importance.to_csv(f"{OUT_DIR}/attack_importance.csv")
    print("### ATTACK_IMPORTANCE"); print(attack_importance.head(25).round(4).to_csv())
    clock_top = supervised.top_features(clock_importance, share=CLOCK_SHARE)
    share = attack_importance[clock_top].sum()
    print(f"### OVERLAP The {len(clock_top)} top clock features have {share:.1%} of the attack model importance.")
else:
    all_results = []
    for row in summary:
        s = split.onset_split(row)
        start = time.time()
        capture = spark.read.parquet(f"{PARQUET_ROOT}/features/attack={s.attack}")
        results, test = supervised.run_fold(train, capture, s, "gbt", TARGET_FPRS, ALERT_RULES, features=features, **GBT)
        results["model"] = f"gbt_{VARIANT}"
        all_results.append(results)
        test.to_parquet(f"{OUT_DIR}/scores/gbt_{VARIANT}_{s.attack}.parquet", index=False)
        print(f"{s.attack}: done in {time.time() - start:.0f} s")
    results = pd.concat(all_results, ignore_index=True)
    results.to_csv(f"{OUT_DIR}/results.csv", index=False)
    print("### RESULTS"); print(results.to_csv(index=False))

+-----------------+------+------+
|           attack|     0|     1|
+-----------------+------+------+
|   active_wiretap|100000|100000|
|         arp_mitm|100000|100000|
|          fuzzing|100000|100000|
|     mirai_botnet|100000|100000|
|          os_scan|100000| 65700|
|       ssdp_flood|100000|100000|
|ssl_renegotiation|100000| 92652|
|          syn_dos|100000|  7038|
|  video_injection|100000|100000|
+-----------------+------+------+

### ATTACK_IMPORTANCE
,importance
f76,0.2062
f79,0.1054
f64,0.0962
f83,0.0806
f62,0.0444
f78,0.0434
f108,0.0417
f61,0.0415
f75,0.0348
f63,0.0327
f29,0.0257
f21,0.0237
f60,0.0213
f97,0.02
f34,0.0153
f58,0.0146
f28,0.0138
f110,0.0132
f59,0.0124
f12,0.0122
f90,0.0094
f0,0.0086
f27,0.008
f38,0.0074
f3,0.0068

### OVERLAP The 10 top clock features have 59.7% of the attack model importance.
CPU times: user 327 ms, sys: 150 ms, total: 477 ms
Wall time: 5min 50s
